# auto-bayesian — Relational quickstart (Kaggle Olist)

This notebook trains an **interpretable Bayesian-network classifier** on the
[Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
dataset, end to end.

Instead of flattening everything by hand, you **declare your tables, keys, and
aggregations**, and `auto-bayesian` joins them into a single modeling frame,
trains a short list of candidate structures (Naive Bayes, TAN, Hill-Climb),
selects the best candidate, tunes a decision threshold, and gives you a model you
can read, score with, and explain.

**Task:** predict whether a delivered order arrives **late** (delivered after the
estimated delivery date) using signals from the customer, the order items, and
the payments — three related tables, joined and aggregated automatically. Late
delivery is a **rare event** (~8% of orders), so we evaluate it the way you
should evaluate any imbalanced classifier: on held-out data, with PR-AUC and a
tuned threshold rather than a misleading fixed 0.5 cutoff.

## 1. Setup

You need:

- `auto-bayesian` installed (from the repo root: `uv sync` or `pip install -e ".[examples]"`).
- `kagglehub` for the dataset download.
- **Kaggle credentials** — `kagglehub` reads `~/.kaggle/kaggle.json` or prompts you
  to log in. Create a token at <https://www.kaggle.com/settings> → *API* → *Create New Token*.

In [1]:
# Uncomment if these are not already installed in your environment:
# %pip install -q kagglehub truststore
# %pip install -q -e "../..[examples]"   # from examples/notebooks/, installs the repo + extras

In [ ]:
import logging
import warnings
from pathlib import Path

import pandas as pd
import truststore

from auto_bayesian import (
    build_project,
    fit_tables,
    generate_explanation,
    materialize_project,
)

# Quiet non-actionable third-party noise: pgmpy/pyro raise SyntaxWarnings from
# their own docstrings on import, and pgmpy logs dtype-inference INFO lines on
# every fit. Neither reflects anything about our data or model.
warnings.filterwarnings("ignore", category=SyntaxWarning)
logging.getLogger("pgmpy").setLevel(logging.WARNING)

# Behind a corporate TLS-inspecting proxy, downloads fail with
# "CERTIFICATE_VERIFY_FAILED: self-signed certificate in certificate chain"
# because the bundled `certifi` store doesn't include the proxy's root CA.
# `truststore` makes Python validate against the OS keychain, which does. It
# must run before any network client (kagglehub) opens a connection.
truststore.inject_into_ssl()

import kagglehub  # noqa: E402  (must import after truststore patches SSL)

pd.set_option("display.max_columns", 50)

In [ ]:
# Note: globally disabling TLS verification (e.g. setting an unverified default
# HTTPS context) is unsafe and unnecessary here — the `truststore` setup in the
# cell above validates against the OS trust store, which works behind corporate
# proxies without turning off certificate checks. This cell is intentionally a
# no-op; you can delete it.

## 2. Download the dataset

`kagglehub.dataset_download` caches the files locally and returns the folder path.

In [4]:
dataset_path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
print("Downloaded to:", dataset_path)
sorted(p.name for p in dataset_path.glob("*.csv"))

Downloaded to: ~/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

## 3. Load the relevant tables

The Olist dataset is genuinely relational: orders link to customers, order items,
and payments. We load the tables we need and let `auto-bayesian` handle the joins.

In [5]:
orders = pd.read_csv(dataset_path / "olist_orders_dataset.csv")
customers = pd.read_csv(dataset_path / "olist_customers_dataset.csv")
order_items = pd.read_csv(dataset_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(dataset_path / "olist_order_payments_dataset.csv")
products = pd.read_csv(dataset_path / "olist_products_dataset.csv")
category_translation = pd.read_csv(dataset_path / "product_category_name_translation.csv")

orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


## 4. Build the target: late delivery

We keep only **delivered** orders with the dates we need, then flag those whose
actual delivery date is later than the estimated one. We also derive two
interpretable calendar features from the purchase timestamp.

In [6]:
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

delivered = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

delivered["is_late"] = (
    delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
).astype(int).astype(str)

delivered["purchase_month"] = delivered["order_purchase_timestamp"].dt.month
delivered["purchase_dow"] = delivered["order_purchase_timestamp"].dt.day_name()

print("Late-delivery rate:")
print(delivered["is_late"].value_counts(normalize=True).round(3))

Late-delivery rate:
is_late
0    0.919
1    0.081
Name: proportion, dtype: float64


## 5. Sample and assemble the tables

Bayesian-network structure search and exact inference are fast, but we subsample
for a snappy notebook run. Lower `SAMPLE_SIZE` if you want it even quicker.

The **root** table (`orders`) holds the target plus the join keys. We also enrich
the order items with a human-readable `product_category` so the model can use the
*latest* category in each order.

In [7]:
SAMPLE_SIZE = 20_000
SEED = 7

sample = delivered.sample(n=min(SAMPLE_SIZE, len(delivered)), random_state=SEED)

orders_root = sample[
    ["order_id", "customer_id", "is_late", "purchase_month", "purchase_dow"]
].copy()

# Keep only the customer columns we actually want as features.
customers_tbl = (
    customers[customers["customer_id"].isin(orders_root["customer_id"])][
        ["customer_id", "customer_state"]
    ].copy()
)

# Enrich order items with an English product category, then restrict to our sample.
products_cat = products.merge(category_translation, on="product_category_name", how="left")
products_cat["product_category"] = (
    products_cat["product_category_name_english"]
    .fillna(products_cat["product_category_name"])
    .fillna("unknown")
)
items = order_items.merge(
    products_cat[["product_id", "product_category"]], on="product_id", how="left"
)
items["product_category"] = items["product_category"].fillna("unknown")
items = items[items["order_id"].isin(orders_root["order_id"])].copy()

payments = order_payments[
    order_payments["order_id"].isin(orders_root["order_id"])
].copy()

print({"orders": len(orders_root), "customers": len(customers_tbl),
       "order_items": len(items), "order_payments": len(payments)})

{'orders': 20000, 'customers': 20000, 'order_items': 22870, 'order_payments': 20906}


## 6. Declare the relational project

This is the heart of `auto-bayesian`: you describe the schema once, and the
library does the joins and aggregations for you.

- `orders → customers` is **one-to-one** (each order has one customer).
- `orders → order_items` is **one-to-many**, so we declare aggregations
  (`count`, `sum`, `mean`, `nunique`, and the time-aware `latest`).
- `orders → order_payments` is **one-to-many** as well.

In [ ]:
project = build_project(
    root_table="orders",
    target_column="is_late",
    positive_label="1",
    test_fraction=0.25,
    numeric_bins=4,
    max_categories=12,
    random_seed=SEED,
    # Late delivery is rare (~8% positives), so we tune for the minority class:
    # supervised binning splits numeric features where the late-rate actually
    # changes, and we pick the candidate by PR-AUC rather than ROC-AUC.
    binning="supervised",
    selection_metric="pr_auc",
    tables=[
        {"name": "orders", "primary_key": "order_id"},
        {"name": "customers", "primary_key": "customer_id"},
        {
            "name": "order_items",
            "primary_key": ["order_id", "order_item_id"],
            "timestamp_column": "shipping_limit_date",
        },
        {"name": "order_payments", "primary_key": ["order_id", "payment_sequential"]},
    ],
    relations=[
        {
            "parent": "orders",
            "child": "customers",
            "parent_key": "customer_id",
            "child_key": "customer_id",
            "kind": "one_to_one",
        },
        {
            "parent": "orders",
            "child": "order_items",
            "parent_key": "order_id",
            "child_key": "order_id",
            "kind": "one_to_many",
            "aggregations": [
                {"op": "count", "name": "n_items"},
                {"column": "price", "op": "sum", "name": "total_item_value"},
                {"column": "freight_value", "op": "mean", "name": "mean_freight"},
                {"column": "seller_id", "op": "nunique", "name": "n_sellers"},
                {"column": "product_category", "op": "latest", "name": "latest_category"},
            ],
        },
        {
            "parent": "orders",
            "child": "order_payments",
            "parent_key": "order_id",
            "child_key": "order_id",
            "kind": "one_to_many",
            "aggregations": [
                {"column": "payment_value", "op": "sum", "name": "total_payment"},
                {"column": "payment_installments", "op": "max", "name": "max_installments"},
                {"op": "count", "name": "n_payments"},
                {"column": "payment_type", "op": "nunique", "name": "n_payment_types"},
            ],
        },
    ],
)
project.task.target_column

'is_late'

## 7. Train and evaluate honestly

Late delivery is **rare** (~8% of orders), so we treat this as the imbalanced
problem it is:

- We **hold out a test set by order** before training, so the reported numbers
  come from rows the model has never seen.
- We score ranking quality with **ROC-AUC** and **PR-AUC** (PR-AUC is the
  honest one for rare positives — read it against the base rate).
- A fixed 0.5 cutoff would label every order "on time" (F1 = 0). Instead the
  model tunes its **decision threshold for F1 on a validation split** during
  `fit`, and `evaluate` reports precision/recall/F1 at that threshold.

In [ ]:
import numpy as np

# Hold out 25% of orders for testing. Child tables are filtered to match, so the
# test set is fully disjoint from training at the order (and customer) level.
rng = np.random.default_rng(SEED)
order_ids = orders_root["order_id"].to_numpy()
is_test = rng.random(len(order_ids)) < 0.25
test_order_ids = set(order_ids[is_test])


def split_tables(*, test: bool) -> dict[str, pd.DataFrame]:
    keep_orders = orders_root[orders_root["order_id"].isin(test_order_ids) == test]
    keep_order_ids = keep_orders["order_id"]
    keep_customer_ids = keep_orders["customer_id"]
    return {
        "orders": keep_orders,
        "customers": customers_tbl[customers_tbl["customer_id"].isin(keep_customer_ids)],
        "order_items": items[items["order_id"].isin(keep_order_ids)],
        "order_payments": payments[payments["order_id"].isin(keep_order_ids)],
    }


train_tables = split_tables(test=False)
test_tables = split_tables(test=True)

# Train on the training tables only; the model tunes its threshold internally.
model = fit_tables(project, train_tables)
report = model.describe()

# Evaluate on the held-out test set using the tuned threshold.
test_frame = materialize_project(project, tables=test_tables)
metrics = model.evaluate(test_frame)

print("Selected structure:", report.selected_candidate)
print(f"Test positive rate: {metrics.positive_rate:.3f}  (late deliveries)")
print(f"ROC AUC:   {metrics.roc_auc:.3f}")
print(f"PR AUC:    {metrics.pr_auc:.3f}  (lift over {metrics.positive_rate:.3f} base rate)")
print(f"Threshold: {metrics.threshold:.3f}  (tuned for F1 on validation)")
print(f"Precision: {metrics.precision:.3f}")
print(f"Recall:    {metrics.recall:.3f}")
print(f"F1:        {metrics.f1:.3f}")

Building tree:   0%|          | 0/78.0 [00:00<?, ?it/s]

Building tree:   0%|          | 0/78.0 [00:00<?, ?it/s]

Selected structure: hill_climb
ROC AUC: 0.686
PR AUC:  0.168
F1:      0.000


## 8. Interpret the model

Unlike a black-box AutoML, the output is a readable network. We can list the
learned dependencies (edges) and read the conditional probability table (CPD) of
the target directly.

In [10]:
print("Learned dependencies:")
for parent, child in report.edges:
    print(f"  {parent} -> {child}")

Learned dependencies:
  customers__customer_state -> is_late
  customers__customer_state -> mean_freight
  is_late -> purchase_month
  mean_freight -> purchase_month
  mean_freight -> total_item_value
  mean_freight -> total_payment
  total_item_value -> latest_category
  total_payment -> max_installments
  total_payment -> total_item_value


In [11]:
# The target's CPD: P(is_late | parents) as a plain probability table.
target_cpd = model.network.get_cpds(report.target_column)
print(target_cpd)

+---------------------------+-----+
| customers__customer_state | ... |
+---------------------------+-----+
| is_late(0)                | ... |
+---------------------------+-----+
| is_late(1)                | ... |
+---------------------------+-----+


## 9. Score new orders

`predict_proba` returns `P(is_late = 1)`; `predict` turns that into a 0/1 label
using the model's **tuned decision threshold** (`report.threshold`) rather than a
fixed 0.5 — the right behavior for a rare positive class. Identical (post-binning)
rows are deduplicated internally, so scoring stays fast on large frames.

In [ ]:
features = test_frame.drop(columns=[report.target_column])

scored = pd.DataFrame({
    "p_late": model.predict_proba(features).round(3),
    "prediction": model.predict(features),
})

print(f"Decision threshold: {report.threshold:.3f}")
print(f"Flagged as late: {int(scored['prediction'].sum())} / {len(scored)} test orders")

# Show the highest-risk orders so the threshold's effect is visible.
scored.sort_values("p_late", ascending=False).head(10)

,p_late,prediction
0,0.087,0
1,0.147,0
2,0.033,0
3,0.073,0
4,0.087,0
5,0.188,0
6,0.030,0
7,0.063,0
8,0.075,0
9,0.033,0


## 10. Visualize the Bayesian network

`generate_explanation` writes a Markdown file with a **Mermaid** diagram of the
learned network plus a plain-language list of the relationships. Each arrow
`A --> B` means *A directly influences B*; the highlighted node is the target.

In [13]:
from IPython.display import Markdown

network_path = generate_explanation(
    model,
    output_path="olist_network.md",
    title="Olist Late-Delivery Network",
)
print("Diagram written to:", network_path)

# JupyterLab renders the Mermaid diagram inline:
Markdown(network_path.read_text())

Diagram written to: examples/notebooks/olist_network.md


# Olist Late-Delivery Network

Predicting **`is_late`** (positive label: `1`).

## Network structure

Each arrow `A --> B` means *A directly influences B*. The highlighted node is the target.

```mermaid
flowchart TD
    n0["customers__customer_state"]
    n1["mean_freight"]
    n2["is_late"]:::target
    n3["total_item_value"]
    n4["latest_category"]
    n5["total_payment"]
    n6["purchase_month"]
    n7["max_installments"]
    n0 --> n2
    n0 --> n1
    n2 --> n6
    n1 --> n6
    n1 --> n3
    n1 --> n5
    n3 --> n4
    n5 --> n7
    n5 --> n3
    classDef target fill:#e74c3c,stroke:#c0392b,color:#fff;
```

## Relationships

- `customers__customer_state` directly predicts `is_late`
- `customers__customer_state` influences `mean_freight`
- `purchase_month` depends on the outcome `is_late`
- `mean_freight` influences `purchase_month`
- `mean_freight` influences `total_item_value`
- `mean_freight` influences `total_payment`
- `total_item_value` influences `latest_category`
- `total_payment` influences `max_installments`
- `total_payment` influences `total_item_value`


## Recap

- We declared **4 related tables** and let `auto-bayesian` handle the joins and
  aggregations.
- It trained and compared **3 Bayesian-network structures** and selected the best.
- The result is **fully interpretable** — readable CPDs plus a Mermaid network diagram.

Next steps: try a different target (e.g. low review score), add `sequence_features`
on a timestamped child table, or switch `task_type` to `next_best_action`.
See the project `README.md` and `DOCUMENTATION.md` for the full reference.